# Regresión logística: probabilidades y regularización

## Objetivo y fundamento
En clasificación binaria, z = b + Σwⱼxⱼ y p = 1/(1+exp(−z)). Aquí Wine tiene tres clases: el modelo multinomial transforma sus puntuaciones con softmax en probabilidades que suman 1. La etiqueta predicha es la de mayor probabilidad. El umbral binario se estudia por separado en el cuaderno 10.

| Concepto | Significado |
|---|---|
| Coeficientes | Efectos en puntuaciones, no efectos causales |
| C pequeño | Regularización más fuerte |
| Escala | Ayuda a comparar penalizaciones de coeficientes |
| Probabilidad | Estimación; no garantía de acierto individual |

## Datos y protocolo

Datos: Wine, incluido en scikit-learn; no requiere descargar CSV. Las 13 mediciones químicas tienen escalas distintas y el objetivo tiene tres clases. Use el entorno `requirements.txt` de esta semana. Reinicie el kernel y ejecute todas las celdas en orden.

Reservamos 25 % para test antes de ajustar transformaciones. La selección usa cinco pliegues estratificados sobre desarrollo y F1 macro (igual peso por clase). Los resultados son de esta muestra, no una garantía universal. Los pliegues no son experimentos independientes ni su desviación es un intervalo de confianza.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.base import clone
datos = load_wine(as_frame=True)
X, y = datos.data, datos.target
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
cv = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X_dev, y_dev))
display(X_dev.describe().T[['mean','std','min','max']])
display(y_dev.value_counts().sort_index().rename('Casos por clase').to_frame())

## Selección con validación cruzada

In [ ]:
pipe = Pipeline([('escala',StandardScaler()),('modelo',LogisticRegression(max_iter=5000))])
grid = {'modelo__C':[.01,.1,1,10,100]}

In [ ]:
busqueda = GridSearchCV(pipe,grid,cv=cv,scoring='f1_macro',return_train_score=True,n_jobs=1)
busqueda.fit(X_dev,y_dev)
tabla = pd.DataFrame(busqueda.cv_results_)[['params','mean_train_score','mean_test_score','std_test_score']]
display(tabla.sort_values('mean_test_score',ascending=False))
print('Configuración elegida por CV:',busqueda.best_params_)
tabla[['mean_train_score','mean_test_score']].plot(marker='o',ylabel='F1 macro',xlabel='Configuración',title='Entrenamiento frente a validación'); plt.show()

## Inspeccionar el modelo con desarrollo

In [ ]:
modelo = busqueda.best_estimator_
p = modelo.predict_proba(X_dev.iloc[:6])
assert np.allclose(p.sum(axis=1),1)
display(pd.DataFrame(p,columns=datos.target_names,index=X_dev.index[:6]))
coef = pd.DataFrame(modelo.named_steps['modelo'].coef_,columns=X.columns,index=datos.target_names)
display(coef)
fig, ax = plt.subplots(figsize=(11,4)); im=ax.imshow(coef,aspect='auto',cmap='coolwarm'); ax.set_xticks(range(X.shape[1]),X.columns,rotation=80); ax.set_yticks(range(3),datos.target_names); ax.set_title('Coeficientes sobre variables estandarizadas'); fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

## Evaluación final
La configuración ya quedó elegida. El test se utiliza ahora para informar desempeño, no para volver a escoger.

In [ ]:
pred = busqueda.predict(X_test)
print(classification_report(y_test,pred,target_names=datos.target_names,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=datos.target_names); plt.title('Test reservado'); plt.show()
assert len(pred)==len(y_test)

## Práctica y discusión
Observe cómo cambia la norma de los coeficientes con C. ¿Por qué su magnitud no demuestra causalidad? ¿Cómo distingue una probabilidad de una etiqueta?

Entregue: tabla de CV, figura comentada y una conclusión que separe selección de evaluación final.

## Galería de rendimiento y clasificación

Ejecute estas celdas después de las anteriores. Las imágenes y gráficos se generan con las predicciones reales de esta ejecución. El test se usa solo para diagnóstico del modelo ya seleccionado. No ajuste parámetros después de observar sus errores.

In [ ]:
modelo_visual = busqueda.best_estimator_

from sklearn.metrics import precision_recall_fscore_support, accuracy_score
# Diagnóstico del modelo ya elegido. No seleccionar otra vez mirando test.
hay_probabilidades = hasattr(modelo_visual,'predict_proba')
prob = modelo_visual.predict_proba(X_test) if hay_probabilidades else modelo_visual.decision_function(X_test)
tipo_salida = 'Probabilidad estimada' if hay_probabilidades else 'Puntuación de decisión (no es probabilidad)'
clases = modelo_visual.classes_
pred_visual = modelo_visual.predict(X_test)
precision, recall, f1, soporte = precision_recall_fscore_support(y_test,pred_visual,labels=clases,zero_division=0)
metricas = pd.DataFrame({'Precisión':precision,'Recall':recall,'F1':f1},index=datos.target_names)
display(metricas.assign(Casos=soporte))
fig, axes = plt.subplots(1,2,figsize=(12,4))
metricas.plot.bar(ax=axes[0],ylim=(0,1.08),rot=0)
axes[0].set(title='Rendimiento por clase en test',ylabel='Métrica')
correctos = pred_visual == np.asarray(y_test)
axes[1].bar(['Aciertos','Errores'],[correctos.sum(),(~correctos).sum()],color=['#08796e','#b05220'])
axes[1].set(title=f'Exactitud en test: {accuracy_score(y_test,pred_visual):.3f}',ylabel='Número de casos')
plt.tight_layout(); plt.show()

# Cada fila es un caso; los errores se muestran primero sin alterar el modelo.
orden = np.argsort(correctos,kind='stable')[:15]
fig, ax = plt.subplots(figsize=(8,6))
im=ax.imshow(prob[orden],vmin=0 if hay_probabilidades else prob.min(),vmax=1 if hay_probabilidades else prob.max(),cmap='Blues',aspect='auto')
ax.set_xticks(range(len(clases)),datos.target_names)
ax.set_yticks(range(len(orden)),[f'Caso {i}: real {np.asarray(y_test)[i]}, pred. {pred_visual[i]}' for i in orden])
for fila,i in enumerate(orden):
    for col in range(len(clases)):
        ax.text(col,fila,f'{prob[i,col]:.2f}',ha='center',va='center',color='white' if im.norm(prob[i,col])>.6 else 'black')
ax.set(title=tipo_salida+'; hasta 15 casos de test, errores primero',xlabel='Clase')
fig.colorbar(im,ax=ax,label=tipo_salida); plt.tight_layout(); plt.show()
if hay_probabilidades: assert np.allclose(prob.sum(axis=1),1)


### Frontera de clasificación en dos dimensiones
Se entrena una copia con solo dos variables sobre desarrollo para visualizar el mecanismo. Su frontera no representa exactamente al modelo completo ni se usa para escoger otra configuración.

In [ ]:
# Modelo auxiliar de dos variables: no es el modelo de 13 variables evaluado arriba.
from matplotlib.colors import ListedColormap
columnas = ['alcohol','color_intensity']  # elegidas antes de inspeccionar test
auxiliar = clone(modelo_visual).fit(X_dev[columnas],y_dev)
rangos = X_dev[columnas].agg(['min','max'])
margen = (rangos.loc['max']-rangos.loc['min'])*.08
gx,gy = np.meshgrid(np.linspace(rangos.loc['min'].iloc[0]-margen.iloc[0],rangos.loc['max'].iloc[0]+margen.iloc[0],180),np.linspace(rangos.loc['min'].iloc[1]-margen.iloc[1],rangos.loc['max'].iloc[1]+margen.iloc[1],180))
malla = pd.DataFrame(np.column_stack([gx.ravel(),gy.ravel()]),columns=columnas)
zona = auxiliar.predict(malla).reshape(gx.shape)
fig,ax=plt.subplots(figsize=(9,5))
ax.contourf(gx,gy,zona,levels=[-.5,.5,1.5,2.5],cmap=ListedColormap(['#d3ebe4','#dbe8fa','#f9e2ce']),alpha=.9)
for clase,color,simbolo in zip([0,1,2],['#08796e','#2468ac','#b05220'],['o','s','^']):
    sel=np.asarray(y_dev)==clase
    ax.scatter(X_dev.loc[sel,columnas[0]],X_dev.loc[sel,columnas[1]],c=color,marker=simbolo,edgecolors='white',label=datos.target_names[clase])
ax.set(xlabel=columnas[0],ylabel=columnas[1],title='Frontera ilustrativa de un modelo auxiliar con dos variables')
ax.legend(title='Clases reales de desarrollo'); plt.tight_layout(); plt.show()
print('El fondo muestra la clase predicha; los símbolos son etiquetas reales. No es una proyección exacta del modelo completo.')

### Interprete las figuras
Identifique las clases más difíciles, describa dos patrones visibles y distinga los gráficos de validación de los de test. Si propone un cambio del modelo, deberá evaluarlo con una nueva separación independiente; no convierta el test observado en un conjunto de selección.